# CKA-RL Walker2D on Kaggle
Attach either the **Walker2D directory as a Kaggle Dataset** or the supplied `Walker2D_CKA-RL.zip`. The setup cell handles both forms and copies/extracts the code into `/kaggle/working/walker2d`. Use a GPU accelerator for the real benchmark.


In [ ]:
from pathlib import Path
import os, shutil, zipfile

input_root = Path('/kaggle/input')
matches = [p for p in input_root.rglob('run_kaggle.sh') if p.parent.joinpath('walker2d_envs.py').exists()]

if not matches:
    zip_matches = []
    for zpath in input_root.rglob('*.zip'):
        try:
            with zipfile.ZipFile(zpath) as zf:
                names = zf.namelist()
                if any(n.endswith('walker2d_envs.py') for n in names) and any(n.endswith('run_kaggle.sh') for n in names):
                    zip_matches.append(zpath)
        except zipfile.BadZipFile:
            pass
    assert len(zip_matches) == 1, f'Expected one Walker2D directory or zip, found dirs={matches}, zips={zip_matches}'
    extract_root = Path('/kaggle/working/_walker2d_extract')
    if extract_root.exists(): shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(zip_matches[0]) as zf:
        zf.extractall(extract_root)
    matches = [p for p in extract_root.rglob('run_kaggle.sh') if p.parent.joinpath('walker2d_envs.py').exists()]

assert len(matches) == 1, f'Expected exactly one Walker2D code directory, found: {matches}'
src = matches[0].parent
dst = Path('/kaggle/working/walker2d')
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(src, dst)
os.chdir(dst)
print('Source:', src)
print('Working directory:', dst)


In [ ]:
!bash run_kaggle.sh setup
!bash run_kaggle.sh sanity


## End-to-end pilot
Run this before a full benchmark. It exercises four dynamics tasks, both Baseline and Combined, and an actual pool merge, but skips expensive retention/survey evaluation. Do **not** interpret its 20k-step returns as final performance.


In [ ]:
!bash run_kaggle.sh pilot


## Main run
After the smoke checks and pilot look healthy, run the full moderate dynamics suite. `all` also trains the from-scratch baselines required for FT/survey metrics. The packaged default is 150k steps/task; the example below uses 200k.


In [ ]:
# Uncomment for the main run.
# !TOTAL_TIMESTEPS=200000 SEEDS='101 102 103' SCRATCH_SEEDS='201 202' bash run_kaggle.sh all
